In [5]:
import sys
import os
sys.path.append(os.path.abspath(".."))
sys.path.append(os.path.abspath("../../source"))
from TDA_Testing import *
import json
import pandas as pd
import pickle
def savepkl(obj, path):
    with open(path, 'wb') as f:
        pickle.dump(obj, f)
def loadpkl(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

# Data Load

In [15]:
# import PD - json, each pds are saved as unnamed array

with open("../../data/json/npc=50_nset=100onepd.json") as f:
    onepd = json.load(f)
with open("../../data/json/npc=50_nset=100twopd.json") as f:
    twopd = json.load(f)

# convert json (saved as unnamed array) to python list
onepdlist=[]
for ii in range(len(onepd)):
    onepdsublist = []
    for jj in range(len(onepd[0])):
        onepddat = np.array(onepd[ii][jj])
        onepdmat = np.transpose( np.resize(onepddat, (3,int(len(onepddat)/3)) ) )
        onepddim1 = onepdmat[onepdmat[:,0]==1,1:]
        onepdsublist.append(onepddim1)
    onepdlist.append(onepdsublist)
    
twopdlist=[]
for ii in range(len(twopd)):
    twopdsublist = []
    for jj in range(len(twopd[0])):
        twopddat = np.array(twopd[ii][jj])
        twopdmat = np.transpose( np.resize(twopddat, (3,int(len(twopddat)/3)) ) )
        twopddim1 = twopdmat[twopdmat[:,0]==1,1:]
        twopdsublist.append(twopddim1)
    twopdlist.append(twopdsublist)

# Simulation with different sample sizes

In [16]:
# simulation parameters
list_npc=[20,40,70,100]
nset=50
nsig=len(sig)
sig=[0.05,0.10,0.15,0.20]
target_sig_index=3 #sig[3] = 0.20 
random.seed(42)
comb = np.array(list(combinations(range(nset), 2)))
ind=random.sample(range(len(comb)),nset)

## Aggregation test

In [5]:
#linear_weight
random.seed(42)
func_weight = function_weight("Poly", poly_order=1)
Agg_linear_opt=np.zeros((len(list_npc),nset))
for n in range(len(list_npc)):
    print(n)
    print(np.sum(Agg_linear_opt==1,axis=1)/nset)
    npc=list_npc[n]
    for jj in range(nset):
        X = onepdlist[target_sig_index][npc*(comb[ind[jj]][0]):npc*(comb[ind[jj]][0]+1)]
        Y = twopdlist[target_sig_index][npc*(comb[ind[jj]][1]):npc*(comb[ind[jj]][1]+1)]
        test_result = Aggtest(X,Y,optimal_bandwidths=True,weight_function=func_weight,Rff_approx=True)
        Agg_linear_opt[n,jj] = test_result

0
[0. 0. 0. 0.]
1
[0.12 0.   0.   0.  ]
2
[0.12 0.26 0.   0.  ]
3
[0.12 0.26 0.66 0.  ]


In [6]:
np.sum(Agg_linear_opt==1,axis=1)/nset 
savepkl(np.sum(Agg_linear_opt==1,axis=1)/nset, '../../results/simulation_results/Circle_results/Circles_Agg_linear_opt_diffsamples.pkl')

In [8]:
#constant_weight
random.seed(42)
func_weight = function_weight("constant")
Agg_constant_opt=np.zeros((len(list_npc),nset))
for n in range(len(list_npc)):
    print(n)
    print(np.sum(Agg_constant_opt==1,axis=1)/nset)
    npc=list_npc[n]
    for jj in range(nset):
        X = onepdlist[target_sig_index][npc*(comb[ind[jj]][0]):npc*(comb[ind[jj]][0]+1)]
        Y = twopdlist[target_sig_index][npc*(comb[ind[jj]][1]):npc*(comb[ind[jj]][1]+1)]
        test_result = Aggtest(X,Y,optimal_bandwidths=True,weight_function=func_weight,Rff_approx=True)
        Agg_constant_opt[n,jj] = test_result

0
[0. 0. 0. 0.]
1
[0.34 0.   0.   0.  ]
2
[0.34 0.56 0.   0.  ]
3
[0.34 0.56 0.84 0.  ]


In [9]:
np.sum(Agg_constant_opt==1,axis=1)/nset 
savepkl(np.sum(Agg_constant_opt==1,axis=1)/nset, '../../results/simulation_results/Circle_results/Circles_Agg_constant_opt_diffsamples.pkl')

In [10]:
#arctan_weight
random.seed(42)
func_weight = function_weight("arctan")
Agg_arctan_opt=np.zeros((len(list_npc),nset))
for n in range(len(list_npc)):
    print(n)
    print(np.sum(Agg_arctan_opt==1,axis=1)/nset)
    npc=list_npc[n]
    for jj in range(nset):
        X = onepdlist[target_sig_index][npc*(comb[ind[jj]][0]):npc*(comb[ind[jj]][0]+1)]
        Y = twopdlist[target_sig_index][npc*(comb[ind[jj]][1]):npc*(comb[ind[jj]][1]+1)]
        test_result = Aggtest(X,Y,optimal_bandwidths=True,weight_function=func_weight,Rff_approx=True)
        Agg_arctan_opt[n,jj] = test_result

0
[0. 0. 0. 0.]
1
[0.12 0.   0.   0.  ]
2
[0.12 0.18 0.   0.  ]
3
[0.12 0.18 0.44 0.  ]


In [11]:
np.sum(Agg_arctan_opt==1,axis=1)/nset 
savepkl(np.sum(Agg_arctan_opt==1,axis=1)/nset, '../../results/simulation_results/Circle_results/Circles_Agg_arctan_opt_diffsamples.pkl')

## PD test 

In [26]:
random.seed(42)
Perm=np.zeros((len(list_npc),nset))
for n in range(len(list_npc)):
    print(n)
    print(np.sum(Perm<0.05,axis=1)/nset)
    npc=list_npc[n]
    for jj in range(nset):
        X = onepdlist[target_sig_index][npc*(comb[ind[jj]][0]):npc*(comb[ind[jj]][0]+1)]
        Y = twopdlist[target_sig_index][npc*(comb[ind[jj]][1]):npc*(comb[ind[jj]][1]+1)]
        T_obs, T_perm, p_val = permutation_test(X, Y, num_permutations=1000)
        Perm[n,jj] = p_val

0
[1. 1. 1. 1.]
1
[0.08 1.   1.   1.  ]
2
[0.08 0.46 1.   1.  ]
3
[0.08 0.46 0.76 1.  ]


In [27]:
savepkl(np.sum(Perm<0.05,axis=1)/nset, '../../results/simulation_results/Circle_results/Circles_PD_result_diffsamples.pkl')
np.sum(Perm<0.05,axis=1)/nset 

array([0.08, 0.46, 0.76, 0.8 ])

## PL test

In [13]:
# convert json (saved as unnamed array) to PL
onepllist=[]
for ii in range(len(onepd)):
    oneplsublist = []
    for jj in range(len(onepd[ii])):
        onepddat = np.array(onepd[ii][jj])
        onepdmat = np.transpose( np.resize(onepddat, (3,int(len(onepddat)/3)) ) )
        onepdmat = [onepdmat[onepdmat[:,0]== np.unique(onepdmat[:,0])[h],1:] for h in range(len(np.unique(onepdmat[:,0])))]
        onecircle_pl = PersLandscapeApprox(dgms=onepdmat, hom_deg=1) # compute persistence landscape
        oneplsublist.append(onecircle_pl)
    onepllist.append(oneplsublist)

twopllist=[]
for ii in range(len(twopd)):
    twoplsublist = []
    for jj in range(len(twopd[ii])):
        twopddat = np.array(twopd[ii][jj])
        twopdmat = np.transpose( np.resize(twopddat, (3,int(len(twopddat)/3)) ) )
        twopdmat = [twopdmat[twopdmat[:,0]== np.unique(twopdmat[:,0])[h],1:] for h in range(len(np.unique(twopdmat[:,0])))]
        twocircle_pl = PersLandscapeApprox(dgms=twopdmat, hom_deg=1) # compute persistence landscape
        twoplsublist.append(twocircle_pl)
    twopllist.append(twoplsublist)

In [17]:
random.seed(42)
PL=np.zeros((len(list_npc),nset))
for n in range(len(list_npc)):
    print(n)
    npc=list_npc[n]
    for jj in range(nset):
        X = onepllist[target_sig_index][npc*(comb[ind[jj]][0]):npc*(comb[ind[jj]][0]+1)]
        Y = twopllist[target_sig_index][npc*(comb[ind[jj]][1]):npc*(comb[ind[jj]][1]+1)]
        PL[n,jj] =permutation_pl_test(X ,Y) # pvalue 

0
1
2
3


In [31]:
savepkl(np.sum(PL<0.05,axis=1)/nset, '../../results/simulation_results/Circle_results/Circles_PL_result_diffsamples.pkl')
np.sum(PL<0.05,axis=1)/nset  

array([0.08, 0.36, 0.7 , 0.78])

In [18]:
np.sum(PL<0.05,axis=1)/nset  

array([0.08, 0.36, 0.7 , 0.78])